In [1]:
#imports 
import urllib.request
import urllib.parse
import json
import re
import time
import os
import pandas as pd



In [6]:
API = "https://gdprhub.eu/api.php"
HEADERS = {"User-Agent": "PrivacyEnforcementResearch/0.1 (jr38088@georgiasouthern.edu)"}

def api_get(params):
    url = API + "?" +urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

titles=[]
for year in range(2018,2027):
    params = {"action": "query", "list": "categorymembers", "cmtitle": f"Category:{year}", "cmlimit": "500", "format": "json"}
    while True:
        data = api_get(params)
        members = data["query"]["categorymembers"]
        titles += [m["title"] for m in members]
        if "continue" in data:
            params["cmcontinue"] = data["continue"]["cmcontinue"]
            time.sleep(1)
        else:
            break
    print(year, "->", len(titles), "cumulative")

titles = sorted(set(t for t in titles if not t.startswith("Category:")))
print(len(titles), "unique decision pages")

2018 -> 16 cumulative
2019 -> 160 cumulative
2020 -> 748 cumulative
2021 -> 1454 cumulative
2022 -> 2012 cumulative
2023 -> 2591 cumulative
2024 -> 3128 cumulative
2025 -> 3609 cumulative
2026 -> 3799 cumulative
3794 unique decision pages


In [10]:
RAW_PATH = "/Users/nic/Documents/MM2/data/raw/gdprhub_raw.jsonl"
os.makedirs(os.path.dirname(RAW_PATH), exist_ok=True)

done = set()
if os.path.exists(RAW_PATH):
    with open(RAW_PATH) as f:
        for line in f:
            done.add(json.loads(line)["title"])

todo = [t for t in titles if t not in done]
print(len(done), "already fetched |", len(todo), "to fetch")

with open(RAW_PATH, "a") as out: 
    for i in range(0, len(todo), 50):
        batch = todo[i:i+50]
        params = {"action": "query", "prop": "revisions", "rvprop": "content",
                  "rvslots": "main", "titles": "|".join(batch), "format": "json"}
        data = api_get(params)
        for page in data["query"]["pages"].values():
            if "revisions" not in page:
                continue
            wikitext = page["revisions"][0]["slots"]["main"]["*"]
            out.write(json.dumps({"title": page["title"], "wikitext":wikitext}) +"\n")
        print(f"{min(i+50, len(todo))}/{len(todo)}")
        time.sleep(2)

98 already fetched | 3696 to fetch
50/3696
100/3696
150/3696
200/3696
250/3696
300/3696
350/3696
400/3696
450/3696
500/3696
550/3696
600/3696
650/3696
700/3696
750/3696
800/3696
850/3696
900/3696
950/3696
1000/3696
1050/3696
1100/3696
1150/3696
1200/3696
1250/3696
1300/3696
1350/3696
1400/3696
1450/3696
1500/3696
1550/3696
1600/3696
1650/3696
1700/3696
1750/3696
1800/3696
1850/3696
1900/3696
1950/3696
2000/3696
2050/3696
2100/3696
2150/3696
2200/3696
2250/3696
2300/3696
2350/3696
2400/3696
2450/3696
2500/3696
2550/3696
2600/3696
2650/3696
2700/3696
2750/3696
2800/3696
2850/3696
2900/3696
2950/3696
3000/3696
3050/3696
3100/3696
3150/3696
3200/3696
3250/3696
3300/3696
3350/3696
3400/3696
3450/3696
3500/3696
3550/3696
3600/3696
3650/3696
3696/3696


In [23]:
def strip_wiki(text):
    text = re.sub(r"\[\[(?:[^|\]]*\|)?([^\]]*)\]\]", r"\1", text)
    text = re.sub(r"'{2,}", "", text)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    return text.strip()

def parse_page(title, wikitext):
    text = strip_wiki(wikitext)

    fields = {}
    for m in re.finditer(r"\|([A-Za-z0-9_]+)\s*=\s*([^|\n]*)", text):
        fields[m.group(1)] = m.group(2).strip()

    def section(name):
        m = re.search(r"===\s*" + name + r"\s*===\s*(.*?)(?=\n===|\n==[^=]|\Z)", text, re.S)
        return m.group(1).strip() if m else None

    art_keys = [k for k in fields if re.fullmatch(r"GDPR_Article_\d+", k)]
    art_keys.sort(key=lambda k: int(k.rsplit("_", 1)[1]))
    articles = [fields[k] for k in art_keys if fields[k]]

    date = fields.get("Date_Decided") or fields.get("Date_Published") or fields.get("Date_Started")
    if fields.get("Date_Decided"):
        date_source = "decided"
    elif fields.get("Date_Published"):
        date_source = "published"
    elif fields.get("Date_Started"):
        date_source = "started"
    else:
        date_source = None

    return {
        "title": title,
        "is_dpa_decision": "DPA_With_Country" in fields or "DPA_Abbrevation" in fields,
        "jurisdiction": fields.get("Jurisdiction"),
        "authority": fields.get("DPA_With_Country"),
        "case_number": fields.get("Case_Number_Name"),
        "type": fields.get("Type"),
        "outcome": fields.get("Outcome"),
        "date": date,
        "date_source": date_source,
        "fine": fields.get("Fine"),
        "currency": fields.get("Currency"),
        "gdpr_articles": "; ".join(articles),
        "facts": section("Facts"),
        "holding": section("Holding"),
    }

records = []
with open(RAW_PATH) as f:
    for line in f:
        d = json.loads(line)
        records.append(parse_page(d["title"], d["wikitext"]))

df = pd.DataFrame(records)
print(df.shape)
print(df["is_dpa_decision"].value_counts())

(3794, 14)
is_dpa_decision
True     2417
False    1377
Name: count, dtype: int64


In [25]:
dpa = df[df["is_dpa_decision"]].drop(columns=["is_dpa_decision"]).reset_index(drop=True)
print(f"kept {len(dpa)} DPA decisions | dropped {len(df) - len(dpa)} court/other pages")

dpa = dpa.replace({"": pd.NA, "None": pd.NA, "n/a": pd.NA, "N/A": pd.NA, "unknown": pd.NA, "Unknown": pd.NA})
dpa["fine_numeric"] = pd.to_numeric(
    dpa["fine"].str.replace(",", "", regex=False).str.extract(r"(\d+\.?\d*)")[0],
    errors="coerce")

dpa["date_parsed"] = pd.to_datetime(
    dpa["date"].str.replace(" ", "", regex=False),
    dayfirst=True, format="mixed", errors="coerce")

dpa.to_csv("/Users/nic/Documents/MM2/data/gdprhub_dpa_decisions.csv", index=False)
print(dpa.shape)
dpa.replace("", pd.NA)[["facts", "holding", "fine", "fine_numeric", "gdpr_articles", "date", "date_parsed", "authority"]].notna().mean().round(3)
dpa[["fine", "currency", "fine_numeric", "date", "date_source"]].head(10)

kept 2417 DPA decisions | dropped 1377 court/other pages
(2417, 15)


,fine,currency,fine_numeric,date,date_source
0,NaN,NaN,NaN,23.02.2024,decided
1,NaN,NaN,NaN,07.09.2022,decided
2,"21,000",EUR,21000.0,18.03.2025,decided
3,NaN,NaN,NaN,07.12.2021,published
4,NaN,NaN,NaN,5. 2. 2020,decided
5,NaN,NaN,NaN,18.11.2021,published
6,NaN,NaN,NaN,15.04.2021,published
7,NaN,NaN,NaN,24.03.2021,published
8,NaN,NaN,NaN,09.04.2021,published
9,NaN,NaN,NaN,09.04.2021,decided


In [26]:
dpa.replace("", pd.NA)[["facts", "holding", "fine", "fine_numeric", "gdpr_articles", "date", "date_parsed", "authority"]].notna().mean().round(3)

facts            0.988
holding          0.993
fine             0.510
fine_numeric     0.509
gdpr_articles    0.947
date             0.990
date_parsed      0.990
authority        1.000
dtype: float64

In [18]:
dpa.replace("", pd.NA)[["facts", "holding", "fine", "gdpr_articles", "date_decided", "authority"]].notna().mean().round(3)

facts            0.988
holding          0.993
fine             1.000
gdpr_articles    0.934
date_decided     0.674
authority        0.996
dtype: float64

In [19]:
dpa["fine"].value_counts().head(15)

fine
|Currency=    721
None          456
10,000         32
2000           31
3000           27
5000           27
20,000         27
1000           24
200,000        23
10000          22
50000          22
5,000          21
50,000         17
40,000         17
15,000         15
Name: count, dtype: int64